# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kiran162005/flyrank-ml/blob/main/work/notebooks/w04_signal_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [2]:
import duckdb
from google.colab import userdata

# Get Hugging Face token from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN is missing. Add HF_TOKEN to Colab Secrets.")

print("HF_TOKEN exists:", bool(HF_TOKEN))
print("HF_TOKEN starts correctly:", HF_TOKEN.startswith("hf_"))

# Create a fresh DuckDB connection
con = duckdb.connect()

# Authenticate DuckDB with Hugging Face
con.execute("DROP SECRET IF EXISTS hf")

con.execute(
    f"""
    CREATE SECRET hf (
        TYPE huggingface,
        TOKEN '{HF_TOKEN.strip()}'
    )
    """
)

# March 2026 Parquet partition
FACT = (
    "read_parquet("
    "'hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/*.parquet'"
    ")"
)

print("DuckDB connected.")
print("FACT defined.")

HF_TOKEN exists: True
HF_TOKEN starts correctly: True
DuckDB connected.
FACT defined.


In [3]:
## 1. Distributions

distribution_check = con.sql(f"""
WITH content AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,
        SUM(gsc_clicks) * 1.0
            / NULLIF(SUM(gsc_impressions), 0) AS ctr,
        AVG(gsc_avg_position) AS avg_position
    FROM {FACT}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
    HAVING SUM(gsc_impressions) > 0
)

SELECT
    COUNT(*) AS n,

    MIN(impressions) AS impressions_min,
    QUANTILE_CONT(impressions, 0.25) AS impressions_p25,
    QUANTILE_CONT(impressions, 0.50) AS impressions_median,
    QUANTILE_CONT(impressions, 0.75) AS impressions_p75,
    QUANTILE_CONT(impressions, 0.90) AS impressions_p90,
    QUANTILE_CONT(impressions, 0.99) AS impressions_p99,
    MAX(impressions) AS impressions_max,

    MIN(clicks) AS clicks_min,
    QUANTILE_CONT(clicks, 0.50) AS clicks_median,
    QUANTILE_CONT(clicks, 0.90) AS clicks_p90,
    QUANTILE_CONT(clicks, 0.99) AS clicks_p99,
    MAX(clicks) AS clicks_max,

    MIN(ctr) AS ctr_min,
    QUANTILE_CONT(ctr, 0.25) AS ctr_p25,
    QUANTILE_CONT(ctr, 0.50) AS ctr_median,
    QUANTILE_CONT(ctr, 0.75) AS ctr_p75,
    QUANTILE_CONT(ctr, 0.90) AS ctr_p90,
    QUANTILE_CONT(ctr, 0.99) AS ctr_p99,
    MAX(ctr) AS ctr_max,

    MIN(avg_position) AS position_min,
    QUANTILE_CONT(avg_position, 0.25) AS position_p25,
    QUANTILE_CONT(avg_position, 0.50) AS position_median,
    QUANTILE_CONT(avg_position, 0.75) AS position_p75,
    QUANTILE_CONT(avg_position, 0.90) AS position_p90,
    QUANTILE_CONT(avg_position, 0.99) AS position_p99,
    MAX(avg_position) AS position_max

FROM content
""").df()

display(distribution_check.T)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,0
n,176738.000000
impressions_min,1.000000
impressions_p25,20.000000
impressions_median,173.000000
impressions_p75,1039.000000
impressions_p90,3930.000000
impressions_p99,21799.780000
impressions_max,617124.000000
clicks_min,0.000000
clicks_median,0.000000


### Distribution observations

The key search-performance variables are not evenly distributed.

- **Impressions** have a heavy right tail: most content items have relatively modest search volume, while a small number receive very large numbers of impressions.
- **Clicks** are also concentrated in a smaller number of high-performing content items.
- **CTR** is bounded near zero and is strongly affected by the amount of search exposure and the mix of queries.
- **Average position** has a wider range, with some pages appearing much lower in search results than others.

Because of these heavy tails, I will avoid interpreting the raw maximum values as representative of a typical page. Medians and percentile ranges are more useful for understanding the data.

For modeling, I will consider transformations such as `log1p(impressions)` and `log1p(clicks)` where appropriate. I will also avoid letting extremely high-volume pages dominate the analysis simply because they have larger raw counts.

These distributions support using robust summary statistics and a validation design that tests whether the model generalizes across clients.

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

### Signal test #1 — Search volume

**Hypothesis:** Pages with higher search impressions have more potential impact if a content issue is identified, so impression volume should be useful for prioritization.

I will compare median CTR across impression-volume buckets. This tests whether volume is associated with a meaningful difference in observed search performance without treating the relationship as causal.

In [4]:
# Signal 1: Search volume

volume_test = con.sql(f"""
WITH content AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) * 1.0
            / NULLIF(SUM(gsc_impressions), 0) AS ctr
    FROM {FACT}
    WHERE gsc_data_available IS TRUE
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) > 0
)
SELECT
    CASE
        WHEN impressions < 100 THEN '<100'
        WHEN impressions < 1000 THEN '100-999'
        WHEN impressions < 10000 THEN '1K-9.9K'
        ELSE '10K+'
    END AS volume_bucket,
    COUNT(*) AS n,
    MEDIAN(impressions) AS median_impressions,
    MEDIAN(ctr) AS median_ctr
FROM content
GROUP BY 1
ORDER BY median_impressions
""").df()

display(volume_test)

,volume_bucket,n,median_impressions,median_ctr
0,<100,75297,13.0,0.000000
1,100-999,56383,317.0,0.000000
2,1K-9.9K,39181,2499.0,0.001931
3,10K+,5877,16156.0,0.001999


### Signal test #2 — CTR

**Hypothesis:** Pages with substantial search exposure but relatively low CTR may have a content or search-snippet opportunity and therefore deserve review.

I will bucket pages by observed CTR and compare their search volume. The purpose is to check whether low-CTR pages still have enough impressions to make them useful refresh candidates.

In [5]:
# Signal 2: CTR opportunity

ctr_test = con.sql(f"""
WITH content AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) * 1.0
            / NULLIF(SUM(gsc_impressions), 0) AS ctr
    FROM {FACT}
    WHERE gsc_data_available IS TRUE
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) > 0
)
SELECT
    CASE
        WHEN ctr < 0.005 THEN '<0.5%'
        WHEN ctr < 0.01 THEN '0.5%-1%'
        WHEN ctr < 0.02 THEN '1%-2%'
        ELSE '2%+'
    END AS ctr_bucket,
    COUNT(*) AS n,
    MEDIAN(impressions) AS median_impressions,
    MEDIAN(ctr) AS median_ctr
FROM content
GROUP BY 1
ORDER BY median_ctr
""").df()

display(ctr_test)

,ctr_bucket,n,median_impressions,median_ctr
0,<0.5%,154529,146.0,0.000000
1,0.5%-1%,12407,853.0,0.006700
2,1%-2%,5457,287.0,0.012987
3,2%+,4345,30.0,0.044444


### Signal test #3 — Average position

**Hypothesis:** Average search position should help distinguish a potential CTR opportunity from pages that simply have very poor visibility. Pages ranking relatively high but receiving low CTR may be more interesting for content/snippet review.

I will compare median CTR across position buckets.

In [6]:
# Signal 3: Average position

position_test = con.sql(f"""
WITH content AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) * 1.0
            / NULLIF(SUM(gsc_impressions), 0) AS ctr,
        AVG(gsc_avg_position) AS avg_position
    FROM {FACT}
    WHERE gsc_data_available IS TRUE
      AND gsc_avg_position IS NOT NULL
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) > 0
)
SELECT
    CASE
        WHEN avg_position <= 3 THEN '1-3'
        WHEN avg_position <= 10 THEN '4-10'
        WHEN avg_position <= 20 THEN '11-20'
        ELSE '20+'
    END AS position_bucket,
    COUNT(*) AS n,
    MEDIAN(avg_position) AS median_position,
    MEDIAN(ctr) AS median_ctr,
    MEDIAN(impressions) AS median_impressions
FROM content
GROUP BY 1
ORDER BY median_position
""").df()

display(position_test)

,position_bucket,n,median_position,median_ctr,median_impressions
0,1-3,17578,2.101857,0.0,128.0
1,4-10,81987,6.022529,0.0,201.0
2,11-20,32204,13.939628,0.0,257.0
3,20+,44969,34.562500,0.0,118.0


### Signal test summary

| Signal | Verdict | Interpretation |
|---|---|---|
| Search volume | CONFIRMED | Higher-volume buckets contain a meaningful number of pages and therefore represent greater potential impact for prioritization. |
| CTR | MIXED | Low CTR can identify possible opportunities, but CTR alone does not show whether the cause is content quality, search intent, SERP features, or ranking. |
| Average position | MIXED | Position helps provide context for CTR, but low CTR at poor positions may simply reflect limited visibility rather than a content problem. |

These verdicts are based on the observed March 2026 data. They are descriptive and directional, not causal.

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*


### Flag-linked signal: Search volume

FlyRank's quick-win logic uses search volume as one of its signals. The assumption is that pages with meaningful search exposure have greater potential impact if an opportunity is identified.

The March 2026 data supports this assumption for prioritization. The volume buckets show that there are 5,823 pages with 10K–99.9K impressions and 54 pages with 100K+ impressions. The 100K+ group also has 54 pages below the 2% CTR threshold.

### Verdict: CONFIRMED

Search volume is a useful prioritization signal because a potential improvement on a high-exposure page could affect more observed search traffic than an equivalent improvement on a very low-volume page.

However, volume alone is not enough to recommend a refresh. It should be combined with signals such as CTR and average position and then reviewed by a human.

### Flag-linked signal: Search volume

FlyRank's quick-win logic uses **search volume** as one of its signals. The assumption is that pages with meaningful search exposure have greater potential impact if an opportunity is identified.

I will test this assumption by comparing pages across impression-volume buckets and measuring the number of pages and their observed CTR.

If higher-volume pages provide a substantial pool of content with measurable CTR opportunity, the volume assumption is supported for prioritization.

This is an observational test. It does not mean that high-volume pages are automatically poor-performing or that changing them will improve performance.

In [7]:
# Flag-linked test: Search volume / quick-win assumption

flag_volume_test = con.sql(f"""
WITH content AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,
        SUM(gsc_clicks) * 1.0
            / NULLIF(SUM(gsc_impressions), 0) AS ctr
    FROM {FACT}
    WHERE gsc_data_available IS TRUE
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) > 0
)

SELECT
    CASE
        WHEN impressions < 100 THEN '<100'
        WHEN impressions < 1000 THEN '100-999'
        WHEN impressions < 10000 THEN '1K-9.9K'
        WHEN impressions < 100000 THEN '10K-99.9K'
        ELSE '100K+'
    END AS volume_bucket,

    COUNT(*) AS n,

    MEDIAN(impressions) AS median_impressions,

    MEDIAN(ctr) AS median_ctr,

    COUNT(*) FILTER (
        WHERE ctr < 0.02
    ) AS low_ctr_pages

FROM content
GROUP BY 1
ORDER BY median_impressions
""").df()

display(flag_volume_test)

,volume_bucket,n,median_impressions,median_ctr,low_ctr_pages
0,<100,75297,13.0,0.000000,71733
1,100-999,56383,317.0,0.000000,55804
2,1K-9.9K,39181,2499.0,0.001931,38994
3,10K-99.9K,5823,16071.0,0.001996,5808
4,100K+,54,131022.5,0.002715,54


### Verdict: CONFIRMED

The data supports search volume as a useful prioritization signal. There are many pages with meaningful search exposure, including 5,823 pages in the 10K–99.9K impression bucket and 54 pages above 100K impressions.

The high-volume groups also contain pages with CTR below 2%, including all 54 pages in the 100K+ bucket. This means high search volume can identify pages where a potential CTR opportunity could have meaningful impact.

However, volume alone does not prove that a page needs a refresh. It should be combined with another signal such as CTR or average position before taking action.

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*



Content teams should use search volume as a prioritization signal rather than treating it as proof that a page needs a refresh. The strongest candidates are high-exposure pages where another signal, such as low CTR relative to search position, also suggests an opportunity.

The audit also shows why a simple rule can make weak picks: low CTR may come from ranking, search intent, SERP features, or query mix rather than stale content. The signals are therefore useful for decision-support, but human review is still required.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.